In [ ]:
import unicodedata
from nltk.sem.logic import Expression
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
import torch
import re
from tqdm.auto import tqdm
from transformers import (
    T5ForConditionalGeneration,
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainer,              
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, 
    EarlyStoppingCallback
)

In [ ]:
#-----------------------------------------------------------
# Preprocess and Evaluation Metrics
#-----------------------------------------------------------

def fol_preprocess(expression):
    replacements = {
        '∀': ' FORALL ', '∃': ' EXISTS ', '¬': 'NOT ', '∧': 'AND',
        '⊕': 'XOR', '∨': 'OR', '→': 'THEN', '↔': 'IFF',
    }
    for symbol, replacement in replacements.items():
        expression = expression.replace(symbol, replacement)
    return expression

def fol_postprocess(text):
    inv_replacements = {
        'FORALL': '∀', 'EXISTS': '∃', 'NOT': '¬', 'AND': '∧',
        'XOR': '⊕', 'OR': '∨', 'THEN': '→', 'IFF': '↔',
    }
    for replacement, symbol in inv_replacements.items():
        text = text.replace(replacement, symbol)
    return text

def nl_preprocess(sentence):
    return unicodedata.normalize('NFKC', sentence.lower())

def to_nltk(expression):
    replacements = {
        '∀': ' all ', '∃': ' exists ', '¬': '-', '∧': '&', '⊕': '!=',
        '∨': '|', '→': '->', '↔': '<->', 'FORALL': ' all ', 'EXISTS': ' exists ',
        'NOT': ' - ', 'AND': '&', 'XOR': '!=', 'OR': '|', 'THEN': '->', 'IFF': '<->',
    }
    for symbol, nltk in replacements.items():
        expression = re.sub(re.escape(symbol), nltk, expression)
    return expression

def check_grammar(expression):
    nltk_expression = to_nltk(expression)
    try:
        parsed_expr = Expression.fromstring(nltk_expression)
        return len(parsed_expr.free()) == 0
    except Exception:
        return False

def predicate_abstraction(expr1, expr2):
    def abstract_independently(expr):
        seen = {}
        counter = 1
        for match in re.finditer(r'\b([a-zA-Z0-9_]+)\(', expr):
            pred = match.group(1)
            if pred not in seen:
                seen[pred] = str(counter)
                counter += 1
        return seen

    pred_map1 = abstract_independently(expr1)
    pred_map2 = abstract_independently(expr2)

    shared_preds = set(pred_map1) & set(pred_map2)
    aligned_map = {}
    next_id = max(len(pred_map1), len(pred_map2)) + 1
    for pred in shared_preds:
        aligned_map[pred] = str(next_id)
        next_id += 1

    def apply_abstraction(expr, local_map):
        combined = {**local_map}
        for pred in aligned_map:
            if pred in combined:
                combined[pred] = aligned_map[pred]
        for pred in sorted(combined, key=len, reverse=True):
            expr = re.sub(r'\b' + re.escape(pred) + r'\(', combined[pred] + '(', expr)
        return expr

    return apply_abstraction(expr1, pred_map1), apply_abstraction(expr2, pred_map2)

def compare_structure(expr1, expr2):
    abstracted_expr1, abstracted_expr2 = predicate_abstraction(expr1, expr2)
    exp1 = to_nltk(abstracted_expr1)
    exp2 = to_nltk(abstracted_expr2)
    try:
        parsed_expr1 = Expression.fromstring(exp1)
        parsed_expr2 = Expression.fromstring(exp2)
        return parsed_expr1 == parsed_expr2
    except Exception:
        return False

In [ ]:
#-----------------------------------------------------------
# Tokenization and Preprocessing
#-----------------------------------------------------------
dataset_path = "WillowNLtoFOL.xlsx"

def tokenize_data(dataset, tokenizer):
    def tokenize_function(examples):
        model_inputs = tokenizer(examples['NL'], max_length=256, truncation=True)
        labels = tokenizer(text_target=examples['FOL'], max_length=256, truncation=True)
        
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return dataset.map(tokenize_function, batched=True)

def load_train_val_test_data(tokenizer):
    dataset = pd.read_excel(dataset_path)
    train_data, test_data = train_test_split(dataset, test_size=0.03, random_state=42)
    train_data, validation_data = train_test_split(train_data, test_size=0.1, random_state=42)

    print(f"Train Dataset size: {len(train_data)}")
    print(f"Validation Dataset size: {len(validation_data)}")
    print(f"Test Dataset size: {len(test_data)}")

    train_data = Dataset.from_pandas(train_data)
    validation_data = Dataset.from_pandas(validation_data)
    test_data = Dataset.from_pandas(test_data)

    train_data = train_data.map(lambda x: {'NL': 'translate English to First-order Logic: ' + nl_preprocess(x['NL_sentence']), 'FOL': fol_preprocess(x['FOL_expression'])})
    validation_data = validation_data.map(lambda x: {'NL': 'translate English to First-order Logic: ' + nl_preprocess(x['NL_sentence']), 'FOL': fol_preprocess(x['FOL_expression'])})
    test_data = test_data.map(lambda x: {'NL': 'translate English to First-order Logic: ' + nl_preprocess(x['NL_sentence']), 'FOL': fol_preprocess(x['FOL_expression'])})

    train_data = tokenize_data(train_data, tokenizer)
    validation_data = tokenize_data(validation_data, tokenizer)
    test_data = tokenize_data(test_data, tokenizer)

    return train_data, validation_data, test_data

In [ ]:
#-----------------------------------------------------------
# Training Setup
#-----------------------------------------------------------
model_name = "google-t5/t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_dataset, validation_dataset, test_dataset = load_train_val_test_data(tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
def model_init():
    return T5ForConditionalGeneration.from_pretrained(model_name, device_map="auto")

def check_tokenization(encodings, tokenizer):
   for i in range(min(5, len(encodings))): 
       print("NL:", tokenizer.decode(encodings['input_ids'][i], skip_special_tokens=True))
       print("FOL:", tokenizer.decode(encodings['labels'][i], skip_special_tokens=True))

check_tokenization(train_dataset, tokenizer)

In [ ]:
def compute_metrics_exact_match(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
        
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    exact_matches = 0
    for pred_text, label_text in zip(decoded_preds, decoded_labels):
        pred = fol_postprocess(pred_text)
        ref = fol_postprocess(label_text)
        if pred.strip() == ref.strip():
            exact_matches += 1
            
    return {"exact_match": exact_matches / len(decoded_labels)}

In [ ]:
# -----------------------------------------------------------
# 1. Hyperparameters
# -----------------------------------------------------------
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
EPOCHS = 2
WEIGHT_DECAY = 1e-2

final_output_dir = './model_t5_base'
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)

# -----------------------------------------------------------
# 2. Training Arguments 
# -----------------------------------------------------------
final_training_args = Seq2SeqTrainingArguments(
    output_dir=final_output_dir,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    
    eval_strategy="steps",
    eval_steps=437,
    save_strategy="steps",
    save_steps=437,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_exact_match",
    greater_is_better=True,
    
    warmup_steps=300,
    adam_beta2=0.999,
    optim="adamw_torch",
    adam_epsilon=1e-8,
    gradient_accumulation_steps=1,
    
    predict_with_generate=True,
    generation_max_length=512, 
    
    logging_dir='./logs_final',
    logging_steps=5,
    disable_tqdm=False,
    report_to="none"
)

# -----------------------------------------------------------
# 3. Initialize and Train
# -----------------------------------------------------------
final_model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

final_trainer = Seq2SeqTrainer(
    model=final_model,
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_exact_match, 
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print(f"--- Starting Final Training (Batch Size: {BATCH_SIZE}, LR: {LEARNING_RATE}) ---")
final_trainer.train()

final_trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)
print(f"Final model saved to {final_output_dir}")